# Trial Exclusion Analysis
We examine the proportion of trials excluded by different exclusion criteria, and how these proportions differ across trial categories `Color`/`BW`/`Noise BG`.

In [1]:
import pandas as pd
import polars as pl
from pymer4.models import glmer
import plotly.io as pio

import config as cnfg
import analysis.helpers.funnels.funnel_config as fcnfg
from analysis.helpers.read_data import read_data
from analysis.helpers.funnels.trial_inclusion import check_trial_inclusion_criteria
from analysis.helpers.visualizations.funnel.category_comparison import category_comparison_figure

pio.renderers.default = "browser"      # or "browser"

Error importing in API mode: ImportError('On Windows, cffi mode "ANY" is only "ABI".')
Trying to import in ABI mode.


### Load Data

In [2]:
loaded_data = read_data(cnfg.OUTPUT_PATH, drop_bad_eye=True, drop_outliers=True)
metadata = loaded_data.metadata
fixations = loaded_data.fixations
actions = loaded_data.actions
idents = loaded_data.identifications
del loaded_data

### Apply Inclusion/Exclusion Criteria

In [3]:
inclusion_criteria = check_trial_inclusion_criteria(
    metadata, fixations, actions, idents,
    min_gaze_coverage=fcnfg.DEFAULT_GAZE_COVERAGE_PERCENT_THRESHOLD,
    min_fixation_rate=fcnfg.DEFAULT_FIXATION_RATE_THRESHOLD,
    bad_actions=fcnfg.DEFAULT_BAD_ACTIONS,
    require_actions=False,
)

# merge with trial category and target count
inclusion_criteria = inclusion_criteria.merge(
    metadata[["subject", "trial", "trial_category"]],
    on=["subject", "trial"],
    how="left"
)

print(f"Overall inclusion rate: {100 * inclusion_criteria['is_valid_trial'].mean():.2f}%")

Overall inclusion rate: 85.00%


### Inclusion by Trial Category
We check which trial category results in more excluded trials, regardless of the exclusion criterion.

In [4]:
inclusion_by_subject = inclusion_criteria.groupby('subject')['is_valid_trial'].agg(['mean', 'std', 'sem'])
inclusion_by_subject["trial_category"] = "ALL"
inclusion_by_subject_and_category = inclusion_criteria.groupby(['subject', 'trial_category'])['is_valid_trial'].agg(['mean', 'std', 'sem']).reset_index()
inclusion_by_category = pd.concat([inclusion_by_subject.reset_index(), inclusion_by_subject_and_category])

inclusion_by_category_subject_average = inclusion_criteria.groupby('trial_category')['is_valid_trial'].agg(['mean', 'std', 'sem']) * 100

In [5]:
trial_category_fig = category_comparison_figure(
    inclusion_by_category.reset_index(),
    categ_col="trial_category",
    title="Trial Inclusion Rate by Trial Category",
    yaxis_title="Inclusion Rate (%)",
    show_distributions=True,
    show_individuals=True,
    show_mean=True,
)

display(inclusion_by_category_subject_average)
trial_category_fig.show()

,mean,std,sem
trial_category,,,
BW,80.00,40.083595,2.587385
COLOR,88.75,31.664097,2.043909
NOISE,86.25,34.509413,2.227573


#### Statistical Analysis: Does trial category predict trial inclusion?
We use a mixed-effects logistic regression model to test whether trial category significantly predicts trial inclusion, while accounting for subject-level variability.<br>
Our model is specified as:
$$logit(P[\text{is\_valid\_trial} = 1]) = \beta_0 + \beta_1 \cdot C(\text{trial\_category}) + (1 | \text{subject})$$

In [6]:
model = glmer(
    formula="is_valid_trial ~ C(trial_category) + (1 | subject)",
    data=pl.from_pandas(inclusion_criteria),
    family="binomial"
)
model.set_factors({
    "subject": (
        pd.Series(inclusion_criteria["subject"].unique())
        .sort_values()
        .map(str, na_action="ignore")
        .to_list()
    ),
    "trial_category": (
        pd.Series(inclusion_criteria["trial_category"].unique())
        .sort_values()
        .tolist()
    ),
})
result = model.fit(summary=True, exponentiate=False)

print("Pairwise Comtrasts:")
display(model.emmeans("trial_category", contrasts="pairwise", p_adjust="tukey"))
result

Pairwise Comtrasts:


contrast,odds_ratio,SE,df,asymp_LCL,asymp_UCL,null,z_ratio,p_value
cat,f64,f64,f64,f64,f64,f64,f64,f64
"""BW / COLOR""",0.487391,0.129465,inf,0.261521,0.90834,1.0,-2.705621,0.018697
"""BW / NOISE""",0.62012,0.156813,inf,0.342835,1.121673,1.0,-1.889638,0.14164
"""COLOR / NOISE""",1.272325,0.358327,inf,0.657563,2.461834,1.0,0.855182,0.668587


GT(_tbl_data=shape: (6, 10)
┌────────────────┬─────────────────┬──────────┬───────────┬───┬──────────┬──────┬──────────┬───────┐
│ rfx            ┆ param           ┆ estimate ┆ conf_low  ┆ … ┆ z_stat   ┆ df   ┆ p_value  ┆ stars │
│ ---            ┆ ---             ┆ ---      ┆ ---       ┆   ┆ ---      ┆ ---  ┆ ---      ┆ ---   │
│ str            ┆ str             ┆ f64      ┆ f64       ┆   ┆ f64      ┆ f64  ┆ str      ┆ str   │
╞════════════════╪═════════════════╪══════════╪═══════════╪═══╪══════════╪══════╪══════════╪═══════╡
│ subject-sd     ┆ (Intercept)     ┆ 0.6339   ┆ null      ┆ … ┆ null     ┆ null ┆ null     ┆ null  │
│ null           ┆ null            ┆ null     ┆ null      ┆ … ┆ null     ┆ null ┆ null     ┆ null  │
│ Fixed Effects: ┆ null            ┆ null     ┆ null      ┆ … ┆ null     ┆ null ┆ null     ┆ null  │
│ null           ┆ (Intercept)     ┆ 1.501037 ┆ 1.010554  ┆ … ┆ 5.998132 ┆ inf  ┆ <.001    ┆ ***   │
│ null           ┆ C(trial_categor ┆ 0.718688 ┆ 0.198067  ┆ … ┆ 2.705621 ┆ inf  ┆ 0.006818 ┆ **    │
│                ┆ y)COLOR         ┆          ┆           ┆   ┆          ┆      ┆          ┆       │
│ null           ┆ C(trial_categor ┆ 0.477842 ┆ -0.017784 ┆ … ┆ 1.889638 ┆ inf  ┆ 0.05881  ┆ .     │
│                ┆ y)NOISE         ┆          ┆           ┆   ┆          ┆      ┆          ┆       │
└────────────────┴─────────────────┴──────────┴───────────┴───┴──────────┴──────┴──────────┴───────┘, _body=<great_tables._gt_data.Body object at 0x000001EE6C528C20>, _boxhead=Boxhead([ColInfo(var='rfx', type=<ColInfoTypeEnum.default: 1>, column_label='Random Effects:', column_align='left', column_width=None), ColInfo(var='param', type=<ColInfoTypeEnum.default: 1>, column_label='', column_align='left', column_width=None), ColInfo(var='estimate', type=<ColInfoTypeEnum.default: 1>, column_label='Estimate', column_align='right', column_width=None), ColInfo(var='conf_low', type=<ColInfoTypeEnum.default: 1>, column_label='CI-low', column_align='right', column_width=None), ColInfo(var='conf_high', type=<ColInfoTypeEnum.default: 1>, column_label='CI-high', column_align='right', column_width=None), ColInfo(var='std_error', type=<ColInfoTypeEnum.default: 1>, column_label='SE', column_align='right', column_width=None), ColInfo(var='z_stat', type=<ColInfoTypeEnum.default: 1>, column_label='Z-stat', column_align='right', column_width=None), ColInfo(var='df', type=<ColInfoTypeEnum.default: 1>, column_label='df', column_align='right', column_width=None), ColInfo(var='p_value', type=<ColInfoTypeEnum.default: 1>, column_label='p', column_align='left', column_width=None), ColInfo(var='stars', type=<ColInfoTypeEnum.default: 1>, column_label='', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x000001EE6C5293A0>, _spanners=Spanners([]), _heading=Heading(title='Formula: glmer(is_valid_trial~C(trial_category)+(1|subject))', subtitle=Md(text='Family: *binomial (link: *default*)*  \n            Number of observations: *720*  \n            Confidence intervals: *parametric*  \n            ---------------------  \n            Log-likelihood: *-290*  \n            AIC: *588* | BIC: *607*  \n            Residual error: *1.0*  \n        '), preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x000001EE6C52AAB0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x000001EE6C52A870>, _source_notes=[Md(text='Signif. codes: *0 *** 0.001 ** 0.01 * 0.05 . 0.1*')], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=['param'], rows=None, mask=None), grpname=None, colname='param', rownum=0, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style='italic', weight=None, stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns=['param'], rows=None, mask=None), grpname=None, colname='param', rownum=1, colnum=None, styles=[CellStyleText(color=None, font=None, size=None

### Exclusion by Criteria
Now we examine the proportion of trials excluded by each criterion.<br>
Here, we don't care about the different trial categories, but rather the overall exclusion rates by each criterion.

In [7]:

inclusion_by_subject["trial_category"] = "ALL"
inclusion_by_subject_and_category = inclusion_criteria.groupby(['subject', 'trial_category'])['is_valid_trial'].agg(['mean', 'std', 'sem']).reset_index()
inclusion_by_category = pd.concat([inclusion_by_subject.reset_index(), inclusion_by_subject_and_category])

inclusion_by_category_subject_average = inclusion_criteria.groupby('trial_category')['is_valid_trial'].agg(['mean', 'std', 'sem']) * 100

In [8]:
inclusion_columns = [c for c in fcnfg.TRIAL_INCLUSION_CRITERIA if c in inclusion_criteria.columns]

inclusion_by_criterion_and_subject = (
    inclusion_criteria
    .groupby('subject')[inclusion_columns]
    .agg(['mean', 'std', 'sem'])
    .T.unstack(0).T
    .reset_index()
    .rename(columns={"level_1": "criterion"})
)

inclusion_by_criterion_average_subject = (
    inclusion_criteria[inclusion_columns + ["is_valid_trial"]]
    .agg(["mean", "std", "sem"])
    .T
    .reset_index()
    .rename(columns={"index": "criterion"})
)
inclusion_by_criterion_average_subject["subject"] = "ALL"

In [9]:
criterion_fig = category_comparison_figure(
    inclusion_by_criterion_and_subject.reset_index(),
    categ_col="criterion",
    title="Trial Inclusion Rate by Trial Category",
    yaxis_title="Inclusion Rate (%)",
    show_distributions=True,
    show_individuals=True,
    show_mean=True,
)

display(100 * inclusion_by_criterion_average_subject.set_index(["subject", "criterion"]))
criterion_fig.show()

mean        std       sem
subject criterion                                                
ALL     gaze_coverage             100.000000   0.000000  0.000000
        fixation_rate             100.000000   0.000000  0.000000
        no_bad_action              88.055556  32.453621  1.209475
        no_miss_with_false_alarm   95.972222  19.674664  0.733231
        is_valid_trial             85.000000  35.731965  1.331652